In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformers import AutoTokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("luerhard/PopBERT")

In [1]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("manifesto-project/manifestoberta-xlm-roberta-56policy-topics-context-2023-1-1", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

sentence = "These principles are under threat."
context = "Human rights and international humanitarian law are fundamental pillars of a secure global system. These principles are under threat. Some of the world's most powerful states choose to sell arms to human-rights abusing states."
# For sentences without additional context, just use the sentence itself as the context.
# Example: context = "These principles are under threat."


/mnt/nvme_storage/git/ytpop/.venv/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


{'201 - Freedom and Human Rights': 90.24, '107 - Internationalism: Positive': 5.88, '105 - Military: Negative': 0.83, '106 - Peace': 0.64, '104 - Military: Positive': 0.54, '202 - Democracy': 0.54, '605 - Law and Order: Positive': 0.18, '305 - Political Authority': 0.16, '103 - Anti-Imperialism': 0.14, '503 - Equality: Positive': 0.14, '109 - Internationalism: Negative': 0.12, '203 - Constitutionalism: Positive': 0.09, '603 - Traditional Morality: Positive': 0.05, '601 - National Way of Life: Positive': 0.04, '602 - National Way of Life: Negative': 0.03, '604 - Traditional Morality: Negative': 0.03, '706 - Non-economic Demographic Groups': 0.03, '101 - Foreign Special Relationships: Positive': 0.02, '406 - Protectionism: Positive': 0.02, '416 - Anti-Growth Economy: Positive': 0.02, '501 - Environmental Protection: Positive': 0.02, '504 - Welfare State Expansion': 0.02, '606 - Civic Mindedness: Positive': 0.02, '705 - Underprivileged Minority Groups': 0.02, '102 - Foreign Special Relati

In [58]:

sentence = "These principles are under threat."
context = "Human rights and international humanitarian law are fundamental pillars of a secure global system."
# For sentences without additional context, just use the sentence itself as the context.
# Example: context = "These principles are under threat."


inputs = tokenizer(sentence,
                   context,
                   return_tensors="pt",
                   max_length=300,  #we limited the input to 300 tokens during finetuning
                   padding="max_length",
                   truncation=True,
                   )

logits = model(**inputs).logits

probabilities = torch.softmax(logits, dim=1).tolist()[0]
probabilities = {model.config.id2label[index]: round(probability * 100, 2) for index, probability in enumerate(probabilities)}
probabilities = dict(sorted(probabilities.items(), key=lambda item: item[1], reverse=True))
print(probabilities)
# {'201 - Freedom and Human Rights': 90.76, '107 - Internationalism: Positive': 5.82, '105 - Military: Negative': 0.66...

predicted_class = model.config.id2label[logits.argmax().item()]
print(predicted_class)
# 201 - Freedom and Human Rights


{'201 - Freedom and Human Rights': 90.31, '107 - Internationalism: Positive': 6.27, '202 - Democracy': 0.56, '605 - Law and Order: Positive': 0.38, '305 - Political Authority': 0.32, '104 - Military: Positive': 0.26, '601 - National Way of Life: Positive': 0.22, '106 - Peace': 0.2, '603 - Traditional Morality: Positive': 0.2, '503 - Equality: Positive': 0.17, '501 - Environmental Protection: Positive': 0.13, '109 - Internationalism: Negative': 0.11, '203 - Constitutionalism: Positive': 0.09, '607 - Multiculturalism: Positive': 0.09, '602 - National Way of Life: Negative': 0.07, '606 - Civic Mindedness: Positive': 0.07, '105 - Military: Negative': 0.06, '416 - Anti-Growth Economy: Positive': 0.06, '705 - Underprivileged Minority Groups': 0.06, '103 - Anti-Imperialism': 0.05, '504 - Welfare State Expansion': 0.04, '706 - Non-economic Demographic Groups': 0.04, '101 - Foreign Special Relationships: Positive': 0.02, '108 - European Community/Union: Positive': 0.02, '303 - Governmental and 

In [60]:
from src.data.processors import TranscriptCleaner

cleaner = TranscriptCleaner()

tokens = cleaner.tokenize(sentence + sentence)
context_tokens = cleaner.tokenize(context + sentence)
print(tokens)
print(context_tokens)

inputs = tokenizer(
    tokens,
    context_tokens,
    is_split_into_words=True,
    return_tensors="pt",
    max_length=300,
    padding="max_length",
    truncation=True,
)

with torch.inference_mode():
    out = model(**inputs)

[['These', 'principles', 'are', 'under', 'threat', '.'], ['These', 'principles', 'are', 'under', 'threat', '.']]
[['Human', 'rights', 'and', 'international', 'humanitarian', 'law', 'are', 'fundamental', 'pillars', 'of', 'a', 'secure', 'global', 'system', '.'], ['These', 'principles', 'are', 'under', 'threat', '.']]


In [44]:
import numpy as np

In [61]:
labels = model.config.id2label

In [62]:
probabilities = torch.softmax(out.logits, dim=1).detach().cpu().numpy()
preds = np.argmax(probabilities, axis=1)

In [63]:
[labels[i] for i in preds]

['201 - Freedom and Human Rights', '501 - Environmental Protection: Positive']

In [37]:

probabilities = torch.softmax(logits, dim=1).tolist()[0]
probabilities = {
    model.config.id2label[index]: round(probability * 100, 2)
    for index, probability in enumerate(probabilities)
}
probabilities = dict(sorted(probabilities.items(), key=lambda item: item[1], reverse=True))
print(probabilities)
# {'201 - Freedom and Human Rights': 90.76, '107 - Internationalism: Positive': 5.82, '105 - Military: Negative': 0.66...

predicted_class = model.config.id2label[logits.argmax().item()]
print(predicted_class)
# 201 - Freedom and Human Rights

TypeError: softmax() received an invalid combination of arguments - got (SequenceClassifierOutput, dim=int), but expected one of:
 * (Tensor input, int dim, torch.dtype dtype, *, Tensor out)
 * (Tensor input, name dim, *, torch.dtype dtype)
